In [17]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
import asyncio
import aiohttp
import os

In [18]:
temps = pl.read_csv('temperature_data.csv')
temps.head()

city,timestamp,temperature,season
str,str,f64,str
"""New York""","""2010-01-01""",-3.521188,"""winter"""
"""New York""","""2010-01-02""",2.573067,"""winter"""
"""New York""","""2010-01-03""",1.628866,"""winter"""
"""New York""","""2010-01-04""",6.615328,"""winter"""
"""New York""","""2010-01-05""",5.544347,"""winter"""


### Этап 1 

#### Задание 1

In [19]:
def moving_stats(df, city=None, window_days=30):
    if city is None:
        df_city = df

    else:
        df_city = df.filter(pl.col('city') == city)
        
    df_city = df_city.with_columns(
        mvavg_30_temp=pl.col('temperature').rolling_mean(window_size=window_days).over('city'),
        mvstd_30_temp=pl.col('temperature').rolling_std(window_size=window_days).over('city')
    )
    return df_city

In [20]:
df_mva = moving_stats(temps)
df_mva[28:]

city,timestamp,temperature,season,mvavg_30_temp,mvstd_30_temp
str,str,f64,str,f64,f64
"""New York""","""2010-01-29""",-0.842146,"""winter""",null,null
"""New York""","""2010-01-30""",5.226067,"""winter""",2.024448,5.639117
"""New York""","""2010-01-31""",2.928476,"""winter""",2.239437,5.542519
"""New York""","""2010-02-01""",-2.055133,"""winter""",2.085163,5.597056
"""New York""","""2010-02-02""",-0.145891,"""winter""",2.026005,5.611406
…,…,…,…,…,…
"""Mexico City""","""2019-12-25""",18.157064,"""winter""",12.416593,4.771792
"""Mexico City""","""2019-12-26""",4.871745,"""winter""",12.155535,4.965841
"""Mexico City""","""2019-12-27""",6.787989,"""winter""",11.777493,4.926935


#### Задание 2

In [21]:
temps = (
    temps.with_columns(
        avg_city_season_temp = pl.mean('temperature').over(["city", "season"]),
        std_city_season_temp = pl.std('temperature').over(["city", "season"])
    )
)
temps.head()

city,timestamp,temperature,season,avg_city_season_temp,std_city_season_temp
str,str,f64,str,f64,f64
"""New York""","""2010-01-01""",-3.521188,"""winter""",-0.07636,5.070526
"""New York""","""2010-01-02""",2.573067,"""winter""",-0.07636,5.070526
"""New York""","""2010-01-03""",1.628866,"""winter""",-0.07636,5.070526
"""New York""","""2010-01-04""",6.615328,"""winter""",-0.07636,5.070526
"""New York""","""2010-01-05""",5.544347,"""winter""",-0.07636,5.070526


#### Задание 3

In [22]:
temps = (
    temps.with_columns(
        is_anomaly_temp = pl.when(
            (pl.col('temperature') > pl.col('avg_city_season_temp') + 2 * pl.col('std_city_season_temp')) | (pl.col('temperature') < pl.col('avg_city_season_temp') - 2 * pl.col('std_city_season_temp'))
        ).then(True).otherwise(False)
    )
)
temps.sample(10)

city,timestamp,temperature,season,avg_city_season_temp,std_city_season_temp,is_anomaly_temp
str,str,f64,str,f64,f64,bool
"""Berlin""","""2011-08-06""",24.656772,"""summer""",19.800311,5.110421,false
"""Mexico City""","""2014-10-04""",17.647883,"""autumn""",15.051212,5.022306,false
"""Mumbai""","""2018-05-12""",30.838673,"""spring""",29.821394,4.858462,false
"""Sydney""","""2012-09-07""",14.299216,"""autumn""",19.833708,5.032501,false
"""Beijing""","""2018-05-19""",17.23982,"""spring""",12.86204,4.968663,false
"""Moscow""","""2019-10-11""",5.230285,"""autumn""",8.155019,4.967951,false
"""Singapore""","""2018-01-29""",23.706824,"""winter""",26.812997,4.718921,false
"""Sydney""","""2016-12-19""",15.430089,"""winter""",12.092239,5.170028,false
"""Cairo""","""2015-04-26""",22.601315,"""spring""",25.287132,4.868904,false


#### Задание 4

##### Polars по умолчанию использует паралелльные процессы и потоки, поэтому для наглядности, я перейду на pandas в этом задании

In [23]:
temps_pd = temps.to_pandas().drop('is_anomaly_temp', axis=1)
temps_pd.head()

,city,timestamp,temperature,season,avg_city_season_temp,std_city_season_temp
0,New York,2010-01-01,-3.521188,winter,-0.07636,5.070526
1,New York,2010-01-02,2.573067,winter,-0.07636,5.070526
2,New York,2010-01-03,1.628866,winter,-0.07636,5.070526
3,New York,2010-01-04,6.615328,winter,-0.07636,5.070526
4,New York,2010-01-05,5.544347,winter,-0.07636,5.070526


In [24]:
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=False)
def check_anomaly(row):
    return (row['temperature'] > row['avg_city_season_temp'] + 2 * row['std_city_season_temp']) or \
           (row['temperature'] < row['avg_city_season_temp'] - 2 * row['std_city_season_temp'])


INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [25]:
%%timeit
temps_pd['def_is_anomaly_temp'] = temps_pd.apply(check_anomaly, axis=1)

215 ms ± 3.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
%%timeit
temps_pd['par_is_anomaly_temp'] = temps_pd.parallel_apply(check_anomaly, axis=1)

138 ms ± 4.08 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Задание 6

In [27]:
from scipy import stats

temps = temps.with_columns(
    non_seasonal_temperature = pl.col('temperature') - pl.mean('temperature').over(['city', 'season'])
)

cities = []

for (city, ), group in temps.group_by('city'):
    group = group.sort('timestamp')

    days = group['timestamp'].dt.epoch('d')
    days = (days - days[0]) / 365.
    
    reg = stats.linregress(days.to_numpy(), group['non_seasonal_temperature'].to_numpy())


    row = {
        'city': city,
        'slope_per_year': reg.slope,
        'pvalue': reg.pvalue,
        'significant': reg.pvalue < 0.05,
        'ci95': 1.96 * reg.stderr
    }
    cities.append(row)

trends = pl.DataFrame(cities)
trends.head()

/tmp/ipykernel_266565/3980608065.py:12: DeprecationWarning: Casting from String to Date is deprecated and will be removed in Polars 2.0.
Use `str.to_date()` instead.
  days = group['timestamp'].dt.epoch('d')


city,slope_per_year,pvalue,significant,ci95
str,f64,f64,f64,f64
"""Sydney""",0.020368,0.484538,0.0,0.057105
"""London""",-0.025574,0.368785,0.0,0.055765
"""New York""",0.039203,0.169708,0.0,0.055947
"""Mexico City""",0.009422,0.742707,0.0,0.056249
"""Berlin""",0.020222,0.482035,0.0,0.056371


In [28]:
from dotenv import load_dotenv
load_dotenv() 

API_KEY = os.getenv('API_WEATHER')
CITY = 'London'

url = f"http://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(f"Погода в {CITY}: {data['main']['temp']}°C")
else:
    print("Ошибка при запросе данных", response.status_code)

Погода в London: 20.06°C


#### Задание 3

In [29]:
cur_season = 'summer'
stats = temps.filter((pl.col('city') == CITY) & (pl.col('season') == cur_season)).select(pl.col('std_city_season_temp'), pl.col('avg_city_season_temp'))[0]
std_, avg_  = stats['std_city_season_temp'].item(), stats['avg_city_season_temp'].item()

if avg_ - 2*std_ <= data['main']['temp'] <=avg_ + 2*std_:
    print("Не аномальная температура")
else:
    print("Аномальная температура")

Не аномальная температура


#### Задание 4

In [30]:
def get_temperature(city):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data['main']['temp']
    else:
        print("Ошибка при запросе данных", response.status_code)
        return None

In [31]:
cities = temps.select(pl.col('city').unique()).to_series()
anom_cities = []
norm_cities = []

for city in cities:
    cur_temp = get_temperature(city)

    stats = temps.filter((pl.col('city') == city) & (pl.col('season') == cur_season)).select(pl.col('std_city_season_temp'), pl.col('avg_city_season_temp'))[0]

    std_, avg_  = stats['std_city_season_temp'].item(), stats['avg_city_season_temp'].item()

    if avg_ - 2*std_ <= cur_temp <=avg_ + 2*std_:
        norm_cities.append(city)
    else:
        anom_cities.append(city)

print("В этих городах аномальная температура:")
print(*anom_cities, sep=', ')
print("В этих городах нормальная температура:")
print(*norm_cities, sep=', ')

В этих городах аномальная температура:
Sydney
В этих городах нормальная температура:
Tokyo, Dubai, New York, Paris, Mumbai, Beijing, Moscow, Rio de Janeiro, Mexico City, London, Singapore, Los Angeles, Cairo, Berlin


#### Задание 5

In [32]:
%%timeit
cities = temps.select(pl.col('city').unique()).to_series()
anom_cities = []
norm_cities = []

for city in cities:
    cur_temp = get_temperature(city)

    stats = temps.filter((pl.col('city') == city) & (pl.col('season') == cur_season)).select(pl.col('std_city_season_temp'), pl.col('avg_city_season_temp'))[0]

    std_, avg_  = stats['std_city_season_temp'].item(), stats['avg_city_season_temp'].item()

    if avg_ - 2*std_ <= cur_temp <=avg_ + 2*std_:
        norm_cities.append(city)
    else:
        anom_cities.append(city)

The slowest run took 4.56 times longer than the fastest. This could mean that an intermediate result is being cached.
2.45 s ± 1.91 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
async def async_get_temperature(city, session):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    async with session.get(url) as response:
        if response.status != 200:
            print(f"Ошибка при запросе {city}:", response.status)   
            return None
        data = await response.json()
        return data['main']['temp']


async def main():
    cities = temps.select(pl.col('city').unique()).to_series().to_list()

    async with aiohttp.ClientSession() as session:
        current = await asyncio.gather(
            *(async_get_temperature(city, session) for city in cities),
            return_exceptions=True
        )
    anom_cities, norm_cities = [], []
    for city, cur_temp in zip(cities, current):        
        if cur_temp is None or isinstance(cur_temp, Exception):
            continue
        stats = temps.filter(
            (pl.col('city') == city) & (pl.col('season') == cur_season)
        ).select('std_city_season_temp', 'avg_city_season_temp')[0]
        std_, avg_ = stats['std_city_season_temp'].item(), stats['avg_city_season_temp'].item()
        (norm_cities if avg_ - 2*std_ <= cur_temp <= avg_ + 2*std_ else anom_cities).append(city)

    return anom_cities, norm_cities

In [34]:
import time 
t0 = time.perf_counter()
await main()
print(f'async: {time.perf_counter() - t0:.2f} с')

async: 0.14 с


##### Лучше здесь использовать асинхронное программирование, так как можно будет эффективно использовать время, а не простаивать, пока дойдет один респонс по API